In [3]:
import numpy as np
import pandas as pd

data = pd.read_csv("data/train.csv").drop("id", axis = 1)
ori = pd.read_csv("data/ori.csv")
test = pd.read_csv("data/test.csv")
df = pd.concat([data, ori], ignore_index=True)

In [4]:
# Remove ID column if it exists
if 'id' in df.columns:
    df = df.drop('id', axis=1)

# Convert date column to datetime and extract d,m,y
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'])
    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month 
    df['year'] = df['date'].dt.year

df.dropna(inplace=True)


In [9]:
from autogluon.tabular import TabularPredictor
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(df, test_size=0.2)

# Initialize and train AutoGluon predictor
predictor = TabularPredictor(
    label='Price',
    eval_metric='root_mean_squared_error',
    path='autogluon_models'  # Directory to store models
).fit(
    train_data,
    time_limit=3600,  # Time limit in seconds
    presets='best_quality',  # Or 'high_quality' for faster training
    # excluded_model_types=['KNN']  # Optional: exclude specific models
)

# Evaluate all models
predictor.leaderboard(test_data, silent=True)

# Get feature importance
predictor.feature_importance(test_data)

# Make predictions
predictions = predictor.predict(test_data)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.12.3
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 23.2.0: Wed Nov 15 21:59:33 PST 2023; root:xnu-10002.61.3~2/RELEASE_ARM64_T8112
CPU Count:          8
Memory Avail:       1.26 GB / 8.00 GB (15.8%)
Disk Space Avail:   95.33 GB / 228.27 GB (41.8%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the data. Then ho


Test Set Metrics:


KeyError: 'num_sold'

In [10]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
print("\nTest Set Metrics:")
print(f"RMSE: {np.sqrt(mean_squared_error(test_data['Price'], predictions)):.2f}")
print(f"MAE: {mean_absolute_error(test_data['Price'], predictions):.2f}")
print(f"R2: {r2_score(test_data['Price'], predictions):.2f}")


Test Set Metrics:
RMSE: 39.07
MAE: 33.86
R2: 0.00


In [11]:
predictor.leaderboard(test_data, silent=True)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,NeuralNetFastAI_BAG_L2,-39.064972,-38.914887,root_mean_squared_error,5.647539,14.870220,532.081064,1.786335,2.036563,26.103582,2,True,16
1,CatBoost_BAG_L2,-39.065466,-38.909408,root_mean_squared_error,3.940906,13.069770,591.774415,0.079702,0.236113,85.796933,2,True,14
2,LightGBM_BAG_L2,-39.067487,-38.912647,root_mean_squared_error,4.018434,13.773831,510.429587,0.157229,0.940174,4.452105,2,True,12
3,LightGBMXT_BAG_L2,-39.067611,-38.910425,root_mean_squared_error,4.143318,14.384336,512.278263,0.282114,1.550678,6.300781,2,True,11
4,WeightedEnsemble_L3,-39.072014,-38.900426,root_mean_squared_error,7.260851,22.253126,750.163194,0.002718,0.001994,0.173496,3,True,17
5,WeightedEnsemble_L2,-39.072069,-38.912422,root_mean_squared_error,2.608248,7.552737,456.439077,0.004626,0.002872,0.308078,2,True,10
6,LightGBM_BAG_L1,-39.072582,-38.914395,root_mean_squared_error,0.154945,1.518436,4.022601,0.154945,1.518436,4.022601,1,True,4
7,CatBoost_BAG_L1,-39.074289,-38.916500,root_mean_squared_error,0.206533,0.998671,381.891187,0.206533,0.998671,381.891187,1,True,6
8,NeuralNetFastAI_BAG_L1,-39.075494,-38.922796,root_mean_squared_error,1.961993,3.248931,66.253377,1.961993,3.248931,66.253377,1,True,8
9,LightGBMXT_BAG_L1,-39.076586,-38.917812,root_mean_squared_error,0.191312,1.303106,3.858245,0.191312,1.303106,3.858245,1,True,3


In [15]:
len(predictions)

55610

In [17]:
pred = predictor.predict(test)

In [18]:
sub = pd.read_csv("data/sample_submission.csv")
sub['Price'] = pred

In [21]:
sub.to_csv("outputs/autogluon.csv", index=False)